<a href="https://colab.research.google.com/github/JiaHaoNCKU/Automated-Numbering-System/blob/main/ANS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
# ==============================================================================
# Automated_Wafer_Numbering_Pipeline_Colab.py
# ver. 2.0.5 (Colab Dedicated Edition)
# [Modifications]:
# - Advanced I/O Customization: Replaced format switches with a dynamic Lambda function
#   (NUMBERING_FORMAT_LAMBDA) in the Main block, allowing complete string format customization (e.g., B001, A_001, 1).
# - Maintained ver. 2.0.3 Universal Region Fix (+ pya.Region()) for absolute text stability.
# - Retained KLayout native Zero-width cutline algorithm for topological integrity.
# - Kept all original bottom-up validation and automated purge engines intact.
# - Avoided adding any unrequested error checks.
# ==============================================================================

# ─── Step 0. Environment Initialization and Package Installation ───
print(" Initializing Colab virtual machine environment and installing core packages...")
!pip install -q klayout

import sys
import os
import math
import json
import numpy as np
import klayout.db as pya
from google.colab import files

GITHUB_REPO_PATH = '/content/Automated-Numbering-System'
MAIN_DIR_PATH = os.path.join(GITHUB_REPO_PATH, 'main')
if MAIN_DIR_PATH not in sys.path:
    sys.path.append(MAIN_DIR_PATH)
if GITHUB_REPO_PATH not in sys.path:
    sys.path.append(GITHUB_REPO_PATH)

try:
    from jsonIO import save_to_json, json_numpy_serializer, load_and_reconstruct_from_json
except ImportError:
    def save_to_json(data, filename):
        with open(filename, 'w') as f:
            json.dump(data, f, default=json_numpy_serializer, indent=4)
    def json_numpy_serializer(obj):
        if isinstance(obj, np.ndarray): return obj.tolist()
        return str(obj)
    def load_and_reconstruct_from_json(filename):
        with open(filename, 'r') as f:
            return json.load(f)

# ==============================================================================
# CORE MODULE 1: GDS TO JSON - Geometric Structure Extraction (Pure KLayout Driven)
# ==============================================================================
def export_cell_to_recursive_dict(layout, cell, memo=None, layers_to_keep=None):
    if memo is None: memo = {}
    if cell.name in memo: return memo[cell.name]

    print(f"    [Processing] Parsing cell geometry structure: {cell.name} ...")
    our_cell_def = { "name": cell.name, "self_shape": None, "subcells": {} }
    memo[cell.name] = our_cell_def
    dbu = layout.dbu

    for layer_idx in layout.layer_indexes():
        info = layout.get_info(layer_idx)
        layer, datatype = info.layer, info.datatype
        if layers_to_keep is not None and (layer, datatype) not in layers_to_keep: continue

        shapes = cell.shapes(layer_idx)
        if not shapes.is_empty():
            polys = []
            for shape in shapes.each():
                dpoly = None
                if shape.is_polygon(): dpoly = shape.dpolygon
                elif shape.is_box(): dpoly = pya.DPolygon(shape.dbox)
                elif shape.is_path(): dpoly = shape.dpath.polygon()

                if dpoly:
                    # Manually construct zero-width cutlines via native KLayout API to preserve Topology
                    pts = [[pt.x, pt.y] for pt in dpoly.each_point_hull()]
                    if dpoly.holes() > 0:
                        for h in range(dpoly.holes()):
                            hole_pts = [[pt.x, pt.y] for pt in dpoly.each_point_hole(h)]
                            if not hole_pts: continue
                            entry_point = pts[-1]
                            hole_start = hole_pts[0]
                            # Cutline In
                            pts.append(entry_point)
                            pts.append(hole_start)
                            # Draw Hole
                            pts.extend(hole_pts)
                            # Close Hole Loop
                            pts.append(hole_start)
                            # Cutline Out
                            pts.append(entry_point)

                    polys.append(pts)

            if len(polys) > 0:
                prim_name = f"{cell.name}_merged_polys_{layer}_{datatype}"
                poly_prim_def = { 'name': prim_name, 'self_shapes': polys, 'subcells': {}, '_gds_layer': [layer, datatype] }
                inst_name = f"{prim_name}_instance"
                our_cell_def['subcells'][inst_name] = {
                    'definition': poly_prim_def,
                    'instances': [{'origin': np.array([0.0, 0.0]), 'rotation': 0.0, 'mirror': False, 'magnification': 1.0}]
                }

    for inst in cell.each_inst():
        ref_cell = inst.cell
        sub_def = export_cell_to_recursive_dict(layout, ref_cell, memo, layers_to_keep)

        trans = inst.dcplx_trans
        rot, mirror, mag = trans.angle, trans.is_mirror(), trans.mag
        base_origin = np.array([trans.disp.x, trans.disp.y])
        instances_list = []

        if inst.is_regular_array():
            cols, rows = inst.na, inst.nb
            vec_a_x, vec_a_y = inst.a.x * dbu, inst.a.y * dbu
            vec_b_x, vec_b_y = inst.b.x * dbu, inst.b.y * dbu

            for r in range(rows):
                for c in range(cols):
                    actual_origin = np.array([base_origin[0] + c * vec_a_x + r * vec_b_x, base_origin[1] + c * vec_a_y + r * vec_b_y])
                    instances_list.append({'origin': actual_origin, 'rotation': rot, 'mirror': mirror, 'magnification': mag})
        else:
            instances_list.append({'origin': base_origin, 'rotation': rot, 'mirror': mirror, 'magnification': mag})

        inst_group_name = f"{ref_cell.name}_instance_group"
        n_conflict = 0
        while inst_group_name in our_cell_def['subcells']:
            n_conflict += 1
            inst_group_name = f"{ref_cell.name}_instance_group_{n_conflict}"

        our_cell_def['subcells'][inst_group_name] = {'definition': sub_def, 'instances': instances_list}

    return our_cell_def

def save_cell_to_json(gds_file_path, cell_name, output_filename, layers_to_keep=None):
    layout = pya.Layout()
    layout.read(gds_file_path)
    target_cell = layout.cell(cell_name)
    recursive_data_structure = export_cell_to_recursive_dict(layout, target_cell, memo={}, layers_to_keep=layers_to_keep)
    save_to_json(recursive_data_structure, output_filename)

# ==============================================================================
# CORE MODULE 2: PROBE HYBRID NUMBERING (Bottom-Up Validation)
# ==============================================================================
def transform_polygon(poly, origin, rotation_deg, mirror, magnification):
    transformed = []
    for pt in poly:
        x_r, y_r = pt[0] * magnification, pt[1] * magnification
        if mirror: y_r = -y_r
        angle_rad = math.radians(rotation_deg)
        x_g = origin[0] + x_r * math.cos(angle_rad) - y_r * math.sin(angle_rad)
        y_g = origin[1] + x_r * math.sin(angle_rad) + y_r * math.cos(angle_rad)
        transformed.append([x_g, y_g])
    return transformed

def transform_point(point, origin, rotation_deg, mirror=False, magnification=1.0):
    x_r, y_r = point[0] * magnification, point[1] * magnification
    if mirror: y_r = -y_r
    angle_rad = math.radians(rotation_deg)
    return [origin[0] + x_r * math.cos(angle_rad) - y_r * math.sin(angle_rad),
            origin[1] + x_r * math.sin(angle_rad) + y_r * math.cos(angle_rad)]

def collect_layer_shapes_recursive(definition, target_layer_num, cur_rot=0.0, cur_mirror=False, cur_mag=1.0):
    results = []
    def_subcells = definition.get('subcells', {})

    for k, sub_item in def_subcells.items():
        sub_def = sub_item.get('definition', {})
        gds_layer = sub_def.get('_gds_layer')
        if gds_layer and gds_layer[0] == target_layer_num:
            for p in sub_def.get('self_shapes', []):
                results.append({'poly': p, 'rot': cur_rot, 'mirror': cur_mirror, 'mag': cur_mag})

    for inst_group_name, group_info in def_subcells.items():
        sub_definition = group_info.get('definition', {})
        if sub_definition.get('_gds_layer'): continue

        for inst in group_info.get('instances', []):
            origin = inst.get('origin', [0.0, 0.0])
            inst_rot, inst_mirror, inst_mag = inst.get('rotation', 0.0), inst.get('mirror', False), inst.get('magnification', 1.0)

            next_mag = cur_mag * inst_mag
            next_mirror = cur_mirror ^ inst_mirror
            next_rot = cur_rot - inst_rot if cur_mirror else cur_rot + inst_rot

            child_results = collect_layer_shapes_recursive(sub_definition, target_layer_num, next_rot, next_mirror, next_mag)
            for cr in child_results:
                t_poly = transform_polygon(cr['poly'], origin, inst_rot, inst_mirror, inst_mag)
                results.append({'poly': t_poly, 'rot': cr['rot'], 'mirror': cr['mirror'], 'mag': cr['mag']})
    return results

# Decoupled Native Vector Font Engine (ver. 2.0.5 Re-verified Robust Solution)
def generate_native_vector_text_polygons(text_str, center, size=10.0):
    generator = pya.TextGenerator.default_generator()
    text_region = generator.text(text_str, 0.001)

    # Standard union operation to extract raw geometry flattening
    flat_polygon_set = text_region + pya.Region()
    if flat_polygon_set.is_empty():
        return []

    bbox = text_region.bbox()
    current_height = bbox.height() if bbox.height() > 0 else 1.0
    current_center_x = bbox.center().x
    current_center_y = bbox.center().y

    scaling_multiplier = size / current_height
    all_polys = []

    for p_int in flat_polygon_set.each():
        dpoly = p_int.to_dtype(1.0)
        hull_pts = [[pt.x, pt.y] for pt in dpoly.each_point_hull()]

        # Zero-width cutline cascade algorithm for hole rendering topology preservation
        if dpoly.holes() > 0:
            for h in range(dpoly.holes()):
                hole_pts = [[pt.x, pt.y] for pt in dpoly.each_point_hole(h)]
                if not hole_pts: continue
                entry_point = hull_pts[-1]
                hole_start = hole_pts[0]
                hull_pts.append(entry_point)
                hull_pts.append(hole_start)
                hull_pts.extend(hole_pts)
                hull_pts.append(hole_start)
                hull_pts.append(entry_point)

        adjusted_poly = []
        for pt in hull_pts:
            adjusted_poly.append([
                (pt[0] - current_center_x) * scaling_multiplier + center[0],
                (pt[1] - current_center_y) * scaling_multiplier + center[1]
            ])
        all_polys.append(adjusted_poly)

    return all_polys

# 🚀 Upgraded Bottom-Up Deep Traversal Scanner
def recursive_scan_wafer_tree(definition, all_collected_probes, accum_origin, accum_rot, accum_mirror, accum_mag):
    def_name = definition.get('name', 'UNKNOWN')
    subcells_dict = definition.get('subcells', {})

    if definition.get('_gds_layer'):
        return False

    layer_81_results = collect_layer_shapes_recursive(definition, 81)
    layer_100_results = collect_layer_shapes_recursive(definition, 100)

    # Base baseline threshold: Self or descendants must simultaneously contain both tracking features
    if not (layer_81_results and layer_100_results):
        return False

    child_is_probe = False

    # 1. Explore child layers first: verify if any descendant entity already qualified as a probe
    for child_inst_name, child_group in subcells_dict.items():
        child_definition = child_group.get('definition', {})
        if child_definition.get('_gds_layer'): continue

        for inst in child_group.get('instances', []):
            origin = inst.get('origin', [0.0, 0.0])
            rotation, mirror, magnification = inst.get('rotation', 0.0), inst.get('mirror', False), inst.get('magnification', 1.0)

            next_mag = accum_mag * magnification
            next_mirror = accum_mirror ^ mirror
            next_rot = accum_rot - rotation if accum_mirror else accum_rot + rotation
            next_origin = transform_point(origin, accum_origin, accum_rot, accum_mirror, accum_mag)

            # If child evaluates to True, it means the child is the real probe; the parent is just a container cell
            if recursive_scan_wafer_tree(child_definition, all_collected_probes, next_origin, next_rot, next_mirror, next_mag):
                child_is_probe = True

    # 2. Evaluation settlement: if any child was already captured, pass this layer and do not label here
    if child_is_probe:
        return True

    # 3. Lowest target interception: no child is a probe, but this cell contains Layer 81 and 100. Target confirmed!
    base_rot = layer_100_results[0]['rot']
    base_mirror = layer_100_results[0]['mirror']
    base_mag = layer_100_results[0]['mag']

    unrotated_verts = []
    for r in layer_100_results:
        for pt in r['poly']:
            angle_rad = math.radians(-base_rot)
            x_unrot = pt[0] * math.cos(angle_rad) - pt[1] * math.sin(angle_rad)
            y_unrot = pt[0] * math.sin(angle_rad) + pt[1] * math.cos(angle_rad)
            if base_mirror: y_unrot = -y_unrot
            unrotated_verts.append([x_unrot / base_mag, y_unrot / base_mag])

    concat_verts = np.array(unrotated_verts)
    min_x, min_y = np.min(concat_verts, axis=0)
    max_x, max_y = np.max(concat_verts, axis=0)

    local_O_height = max_y - min_y
    if local_O_height < 5.0: local_O_height = 35.0
    unrotated_center = [(min_x + max_x) / 2.0, (min_y + max_y) / 2.0]

    x_c, y_c = unrotated_center[0] * base_mag, unrotated_center[1] * base_mag
    if base_mirror: y_c = -y_c
    a_rad = math.radians(base_rot)
    relative_center = [x_c * math.cos(a_rad) - y_c * math.sin(a_rad), x_c * math.sin(a_rad) + y_c * math.cos(a_rad)]

    global_center = transform_point(relative_center, accum_origin, accum_rot, accum_mirror, accum_mag)

    all_collected_probes.append({
        'type': def_name,
        'global_center': global_center,
        'local_size': local_O_height,
        'base_rot': base_rot,
        'base_mirror': base_mirror,
        'base_mag': base_mag,
        'final_origin': global_center,
        'final_rot': accum_rot,
        'final_mirror': accum_mirror,
        'final_mag': accum_mag
    })
    return True

def execute_safe_probe_purge(definition, current_cell_name, target_layer_num=100):
    subcells_dict = definition.get('subcells', {})
    keys_to_delete = []

    if current_cell_name != "WAFER":
        for k, sub_item in subcells_dict.items():
            sub_def = sub_item.get('definition', {})
            gds_layer = sub_def.get('_gds_layer')
            if (gds_layer and gds_layer[0] == target_layer_num) or (sub_def.get('name') in ["O", "0"]):
                keys_to_delete.append(k)

        if keys_to_delete:
            print(f"🧹 [Localized Purge] Erasing {len(keys_to_delete)} old Layer 100 markers/cells from probe [{current_cell_name}]...")
            for k in keys_to_delete: del subcells_dict[k]

    for child_inst_name, child_group in subcells_dict.items():
        child_definition = child_group.get('definition', {})
        if not child_definition.get('_gds_layer'):
            execute_safe_probe_purge(child_definition, child_definition.get('name', 'UNKNOWN'), target_layer_num)

def process_hybrid_wafer_numbering(json_path, output_json_path, sort_mode="CARTESIAN", radial_center=(0.0, 0.0), format_fn=None):
    print(f" Loading wafer layout JSON: {json_path}")
    data = load_and_reconstruct_from_json(json_path)
    all_collected_probes = []

    recursive_scan_wafer_tree(data, all_collected_probes, accum_origin=[0.0, 0.0], accum_rot=0.0, accum_mirror=False, accum_mag=1.0)

    grouped_probes = {}
    for probe in all_collected_probes:
        grouped_probes.setdefault(probe['type'], []).append(probe)

    subcells_dict = data.get('subcells', {})

    print(f"\n [Serialization Stage] Injecting new serial numbers directly via Mode: [{sort_mode}]...")
    for p_type, probes_list in grouped_probes.items():
        if sort_mode == "CARTESIAN":
            probes_list.sort(key=lambda p: (-round(p['global_center'][1], 1), round(p['global_center'][0], 1)))
        elif sort_mode == "RADIAL":
            def calculate_polar_clockwise_12(p_item):
                dx, dy = p_item['global_center'][0] - radial_center[0], p_item['global_center'][1] - radial_center[1]
                angle = math.atan2(dx, dy)
                if angle < -0.175: angle += 2 * math.pi
                return (angle, math.hypot(dx, dy))
            probes_list.sort(key=calculate_polar_clockwise_12)

        print(f"    Assigning independent serial numbers for probe model [{p_type}] (Total: {len(probes_list)} probes)...")
        for idx, probe in enumerate(probes_list):
            serial_number = idx + 1

            # Execute decoupled dynamic string formatting callback
            if format_fn is not None:
                constructed_label = format_fn(serial_number)
            else:
                constructed_label = f"{serial_number}"

            # Calling decoupled vector text converter
            digit_polys = generate_native_vector_text_polygons(constructed_label, center=[0.0, 0.0], size=probe['local_size'])

            for p_idx, poly_verts in enumerate(digit_polys):
                prim_name = f"wafer_text_{p_type}_{serial_number}_p{p_idx}"
                actual_text_rot = probe['final_rot'] - probe.get('base_rot', 0.0) if probe['final_mirror'] else probe['final_rot'] + probe.get('base_rot', 0.0)
                actual_text_mirror = probe['final_mirror'] ^ probe.get('base_mirror', False)

                subcells_dict[f"{prim_name}_instance_group"] = {
                    'definition': {'name': prim_name, 'self_shapes': [poly_verts], 'subcells': {}, '_gds_layer': [100, 0]},
                    'instances': [{
                        'origin': probe['final_origin'], 'rotation': actual_text_rot,
                        'mirror': actual_text_mirror, 'magnification': probe['final_mag'] * probe.get('base_mag', 1.0)
                    }]
                }
        print(f"    ✅ Successfully completed native vector height-matched numbering for model [{p_type}].")

    execute_safe_probe_purge(data, current_cell_name="WAFER", target_layer_num=100)
    save_to_json(data, output_json_path)

# ==============================================================================
# CORE MODULE 3: JSON TO GDS - Layout Reconstruction (Pure 1.4.2 Baseline)
# ==============================================================================
def create_gds_cell(layout, cell_definition_json, existing_cells):
    cell_name = cell_definition_json.get('name', 'UNKNOWN_CELL')
    if cell_name in existing_cells: return existing_cells[cell_name]

    pya_cell = layout.create_cell(cell_name)
    existing_cells[cell_name] = pya_cell

    if "wafer_text_" in cell_name: layer = layout.layer(100, 0)
    else:
        orig = cell_definition_json.get('_gds_layer')
        layer = layout.layer(int(orig[0]), int(orig[1])) if orig else layout.layer(100, 0)

    for poly_verts in cell_definition_json.get('self_shapes') or []:
        pya_cell.shapes(layer).insert(pya.DPolygon([pya.DPoint(p[0], p[1]) for p in poly_verts]))

    for sub_group in cell_definition_json.get('subcells', {}).values():
        if sub_cell_def_json := sub_group.get('definition'):
            pya_sub_cell = create_gds_cell(layout, sub_cell_def_json, existing_cells)
            for inst in sub_group.get('instances', []):
                trans = pya.DCplxTrans(inst.get('magnification', 1.0), inst.get('rotation', 0.0),
                                       inst.get('mirror', False), inst.get('origin', [0.0, 0.0])[0], inst.get('origin', [0.0, 0.0])[1])
                pya_cell.insert(pya.DCellInstArray(pya_sub_cell.cell_index(), trans))
    return pya_cell

def convert_json_to_gds(input_json_path, output_gds_path, top_cell_name="TOP"):
    layout = pya.Layout()
    layout.dbu = 0.001
    gds_top_cell = layout.create_cell(top_cell_name)
    pya_main_cell = create_gds_cell(layout, load_and_reconstruct_from_json(input_json_path), {})
    gds_top_cell.insert(pya.DCellInstArray(pya_main_cell.cell_index(), pya.DCplxTrans(1.0, 0.0, False, 0.0, 0.0)))
    layout.write(output_gds_path)

# ==============================================================================
# ─── Step 4. Monolithic Execution Pipeline (Pipeline Main) Controls
# ==============================================================================
if __name__ == "__main__":
    target_cell_name = "WAFER"
    LOCAL_WORKSPACE_DIR = "/content"

    # ==========================================================================
    # 🎛️ PIPELINE RUNTIME FUNCTIONAL SWITCHES (I/O 控制區)
    # ==========================================================================
    CHOSEN_SORT_MODE = "RADIAL"           # 排序模式選項: "RADIAL" 或 "CARTESIAN"
    WAFER_CENTER_COORDS = (0.0, 0.0)      # 晶圓中心座標（極座標排序時使用）

    # 🚀 在此直接透過 Lambda 函式客製化您的編號字串格式！
    # 範例一 (您要的 B001):   lambda sn: f"B{sn:03d}"
    # 範例二 (您要的 A_001):  lambda sn: f"A_{sn:03d}"
    # 範例三 (回歸標準純數字): lambda sn: f"{sn}"
    NUMBERING_FORMAT_LAMBDA = lambda sn: f"B-{sn:03d}"

    # Colab Dynamic Runtime Upload Trigger
    print("🌐 Please upload your target layout GDS file (e.g. Probes_test.GDS)...")
    uploaded_dict = files.upload()
    if not uploaded_dict:
        raise FileNotFoundError("Pipeline execution aborted: No file uploaded.")

    uploaded_gds_name = list(uploaded_dict.keys())[0]
    gds_file_path = os.path.join(LOCAL_WORKSPACE_DIR, uploaded_gds_name)

    mid_raw_json = os.path.join(LOCAL_WORKSPACE_DIR, f"{target_cell_name}.json")
    mid_numbered_json = os.path.join(LOCAL_WORKSPACE_DIR, f"{target_cell_name}_numbered.json")
    final_output_gds = os.path.join(LOCAL_WORKSPACE_DIR, f"{target_cell_name}_numbered.gds")

    print(f"✅ Successfully loaded Colab file: {uploaded_gds_name}. Launching automated runtime pipeline...")

    # Phase 2: Extraction (Driven by Pure KLayout Core to extract 1.6.3 structure)
    allowed_layers = [(81, 0), (100, 0)]
    save_cell_to_json(gds_file_path, target_cell_name, mid_raw_json, layers_to_keep=allowed_layers)
    print("➔ [Phase 2] Geometry boundary extraction to raw JSON completed.")

    # Phase 3: Numbering & Safe Purge (Strictly 1.6.3 Functional Specification)
    if os.path.exists(mid_raw_json):
        process_hybrid_wafer_numbering(mid_raw_json, mid_numbered_json,
                                       sort_mode=CHOSEN_SORT_MODE,
                                       radial_center=WAFER_CENTER_COORDS,
                                       format_fn=NUMBERING_FORMAT_LAMBDA)
        print("➔ [Phase 3] Serial number assignment and dynamic coordinate matrix cascade completed.")
    else:
        raise FileNotFoundError(f"Intermediary raw JSON not found at: {mid_raw_json}")

    # Phase 4: Reconstruction (Strictly 1.6.3 Functional Specification)
    if os.path.exists(mid_numbered_json):
        convert_json_to_gds(mid_numbered_json, final_output_gds, top_cell_name="TOP")
        print("➔ [Phase 4] KLayout database compilation and native physical stream-out completed.")
    else:
        raise FileNotFoundError(f"Intermediary numbered JSON not found at: {mid_numbered_json}")

    print(f"\nAutomated pipeline finished execution! Target GDS generated at: {final_output_gds}")
    print(" Monolithic wafer numbering pipeline workflow accomplished successfully. Triggering dynamic download...")

    # Colab Dynamic Runtime Download Trigger
    files.download(final_output_gds)

 Initializing Colab virtual machine environment and installing core packages...
🌐 Please upload your target layout GDS file (e.g. Probes_test.GDS)...


Saving WAll_structure.GDS to WAll_structure (12).GDS
✅ Successfully loaded Colab file: WAll_structure (12).GDS. Launching automated runtime pipeline...
    [Processing] Parsing cell geometry structure: WAFER ...
    [Processing] Parsing cell geometry structure: Cable_C68-L14 ...
    [Processing] Parsing cell geometry structure: 64ch_FC_connector_v3 ...
    [Processing] Parsing cell geometry structure: Bondpad_70um2 ...
    [Processing] Parsing cell geometry structure: ATLAS_Neuro ...
    [Processing] Parsing cell geometry structure: 64ch_FC_interconnect_8-8_v2 ...
    [Processing] Parsing cell geometry structure: TEXT$64 ...
    [Processing] Parsing cell geometry structure: TEXT$65 ...
    [Processing] Parsing cell geometry structure: ARC$5 ...
    [Processing] Parsing cell geometry structure: TEXT$66 ...
    [Processing] Parsing cell geometry structure: TEXT$67 ...
    [Processing] Parsing cell geometry structure: TEXT$68 ...
    [Processing] Parsing cell geometry structure: TEXT$69 .

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>